Install Dependencies

In [1]:
!pip install pandas numpy scikit-learn sentence-transformers faiss-cpu gradio matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 28.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import json
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sentence_transformers import SentenceTransformer

import matplotlib.pyplot as plt

Load Round 1 Outputs

In [3]:
from google.colab import files

uploaded = files.upload()

Saving hundred_checkpoints.json to hundred_checkpoints.json
Saving persona.json to persona.json
Saving topic_checkpoints.json to topic_checkpoints.json


In [4]:
from google.colab import files

uploaded = files.upload()

Saving conversations.csv to conversations.csv


In [5]:
with open("/content/persona.json") as f:
    persona = json.load(f)

with open("/content/topic_checkpoints.json") as f:
    topic_checkpoints = json.load(f)

with open("/content/hundred_checkpoints.json") as f:
    hundred_checkpoints = json.load(f)

print("Loaded successfully")

Loaded successfully


PART 1
Persona Drift Detector

Load Dataset

In [6]:
df = pd.read_csv("/content/conversations.csv", header=None)

df.columns=["conversation"]

print(df.shape)

(11001, 1)


Create Day Wise Messages

In [7]:
daily_messages = []

for day,row in df.iterrows():

    text = str(row["conversation"])

    msgs = re.findall(
        r'User\s*\d+:\s*(.*?)(?=User\s*\d+:|$)',
        text,
        flags=re.S
    )

    daily_messages.append({
        "day": day+1,
        "messages": msgs
    })

print("Days:",len(daily_messages))

Days: 11001


Simple Mood Detector

In [8]:
positive_words = [
    "happy",
    "excited",
    "great",
    "awesome",
    "love",
    "wonderful"
]

negative_words = [
    "sad",
    "upset",
    "frustrated",
    "angry",
    "stress",
    "worried"
]

In [9]:
def detect_mood(messages):

    text = " ".join(messages).lower()

    pos = sum(
        word in text
        for word in positive_words
    )

    neg = sum(
        word in text
        for word in negative_words
    )

    if pos > neg:
        return "positive"

    elif neg > pos:
        return "negative"

    return "neutral"

Persona Drift Timeline

In [10]:
timeline = []

for day_data in daily_messages:

    mood = detect_mood(
        day_data["messages"]
    )

    timeline.append({
        "day": day_data["day"],
        "mood": mood
    })

timeline[:10]

[{'day': 1, 'mood': 'positive'},
 {'day': 2, 'mood': 'positive'},
 {'day': 3, 'mood': 'positive'},
 {'day': 4, 'mood': 'positive'},
 {'day': 5, 'mood': 'positive'},
 {'day': 6, 'mood': 'positive'},
 {'day': 7, 'mood': 'positive'},
 {'day': 8, 'mood': 'positive'},
 {'day': 9, 'mood': 'positive'},
 {'day': 10, 'mood': 'positive'}]

Detect Drift

In [11]:
drifts = []

for i in range(1,len(timeline)):

    if timeline[i]["mood"] != timeline[i-1]["mood"]:

        drifts.append({

            "from_day":
            timeline[i-1]["day"],

            "to_day":
            timeline[i]["day"],

            "old":
            timeline[i-1]["mood"],

            "new":
            timeline[i]["mood"]
        })

drifts[:10]

[{'from_day': 30, 'to_day': 31, 'old': 'positive', 'new': 'neutral'},
 {'from_day': 31, 'to_day': 32, 'old': 'neutral', 'new': 'positive'},
 {'from_day': 42, 'to_day': 43, 'old': 'positive', 'new': 'neutral'},
 {'from_day': 44, 'to_day': 45, 'old': 'neutral', 'new': 'positive'},
 {'from_day': 47, 'to_day': 48, 'old': 'positive', 'new': 'neutral'},
 {'from_day': 48, 'to_day': 49, 'old': 'neutral', 'new': 'positive'},
 {'from_day': 149, 'to_day': 150, 'old': 'positive', 'new': 'negative'},
 {'from_day': 150, 'to_day': 151, 'old': 'negative', 'new': 'positive'},
 {'from_day': 177, 'to_day': 178, 'old': 'positive', 'new': 'negative'},
 {'from_day': 178, 'to_day': 179, 'old': 'negative', 'new': 'positive'}]

Drift Trigger Detector

In [12]:
from collections import Counter

def extract_trigger(messages):

    words = []

    for m in messages:

        tokens = re.findall(
            r'\b[a-zA-Z]{4,}\b',
            m.lower()
        )

        words.extend(tokens)

    common = Counter(words).most_common(5)

    return [x[0] for x in common]

In [13]:
for d in drifts[:10]:

    msgs = daily_messages[
        d["to_day"]-1
    ]["messages"]

    d["trigger"] = extract_trigger(msgs)

drifts[:10]

[{'from_day': 30,
  'to_day': 31,
  'old': 'positive',
  'new': 'neutral',
  'trigger': ['band', 'like', 'drive', 'just', 'from']},
 {'from_day': 31,
  'to_day': 32,
  'old': 'neutral',
  'new': 'positive',
  'trigger': ['that', 'thanks', 'love', 'really', 'work']},
 {'from_day': 42,
  'to_day': 43,
  'old': 'positive',
  'new': 'neutral',
  'trigger': ['that', 'actually', 'would', 'interesting', 'talk']},
 {'from_day': 44,
  'to_day': 45,
  'old': 'neutral',
  'new': 'positive',
  'trigger': ['that', 'want', 'cool', 'sure', 'love']},
 {'from_day': 47,
  'to_day': 48,
  'old': 'positive',
  'new': 'neutral',
  'trigger': ['that', 'what', 'think', 'favorite', 'them']},
 {'from_day': 48,
  'to_day': 49,
  'old': 'neutral',
  'new': 'positive',
  'trigger': ['love', 'always', 'read', 'food', 'favorite']},
 {'from_day': 149,
  'to_day': 150,
  'old': 'positive',
  'new': 'negative',
  'trigger': ['what', 'play', 'games', 'that', 'like']},
 {'from_day': 150,
  'to_day': 151,
  'old': 'negat

PART 2
Offline Intent Classifier(most important section)


Training Data

In [14]:
training_data = [

("remind me tomorrow", "reminder"),
("set an alarm", "reminder"),

("i feel sad", "emotional-support"),
("i am depressed", "emotional-support"),

("send email to manager", "action-item"),
("book a ticket", "action-item"),

("how are you", "small-talk"),
("good morning", "small-talk"),

("random statement", "unknown")
]

Train Model

In [15]:
texts = [x[0] for x in training_data]

labels = [x[1] for x in training_data]

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(texts)

clf = LogisticRegression()

clf.fit(X,labels)

LogisticRegression()

Prediction Function

In [32]:
def predict_intent(text):

    x = vectorizer.transform([text])

    pred = clf.predict(x)[0]

    return pred

Test

In [33]:
predict_intent(
    "please remind me tonight"
)

np.str_('reminder')

PART 3
Conflict Resolution RAG

Example Contradictions

In [18]:
sister_chunks = [

{
"text":
"My sister lives in Texas",
"day":2,
"emotion":1
},

{
"text":
"My sister moved to California",
"day":7,
"emotion":2
},

{
"text":
"I had an argument with my sister",
"day":8,
"emotion":5
}
]

Ranking Function

In [19]:
def score_chunk(chunk):

    recency = chunk["day"]

    emotion = chunk["emotion"]

    return recency + emotion

Resolver

In [20]:
def resolve_conflict(chunks):

    ranked = sorted(
        chunks,
        key=score_chunk,
        reverse=True
    )

    answer = []

    for c in ranked:

        answer.append(c["text"])

    return answer

In [21]:
resolve_conflict(
    sister_chunks
)

['I had an argument with my sister',
 'My sister moved to California',
 'My sister lives in Texas']

Contradiction Detector

In [22]:
def contradiction_detector(chunks):

    locations = []

    for c in chunks:

        if "Texas" in c["text"]:
            locations.append("Texas")

        if "California" in c["text"]:
            locations.append("California")

    return len(set(locations)) > 1

In [23]:
contradiction_detector(
    sister_chunks
)

True

PART 4
Chatbot Demo

In [24]:
import gradio as gr

In [37]:
def chatbot(question):

    q = question.lower()

    # habits
    if "habit" in q or "habits" in q:
        return "\n".join(persona["habits"][:5])

    # communication style
    elif (
        "talk" in q
        or "communicate" in q
        or "communication" in q
        or "speak" in q
    ):
        return "\n".join(persona["communication_style"])

    # personality
    elif (
        "personality" in q
        or "traits" in q
        or "trait" in q
    ):
        return "\n".join(
            persona["personality_traits"][:5]
        )

    # personal facts
    elif (
        "fact" in q
        or "facts" in q
        or "about user" in q
    ):
        return "\n".join(
            persona["personal_facts"][:5]
        )

    # sister conflict resolver
    elif "sister" in q:

        contradiction = contradiction_detector(
            sister_chunks
        )

        response = ""

        if contradiction:
            response += (
                "⚠ Contradictions detected\n\n"
            )

        response += "\n".join(
            resolve_conflict(
                sister_chunks
            )
        )

        return response

    # mood drift
    elif (
        "drift" in q
        or "mood" in q
        or "timeline" in q
    ):

        output = ""

        for item in timeline[:10]:

            output += (
                f"Day {item['day']} : "
                f"{item['mood']}\n"
            )

        return output

    return "No relevant information found."

In [42]:
demo = gr.Interface(

    fn=chatbot,

    inputs="text",

    outputs="text",

    title="Adaptive Persona Engine"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2c1a69e9b996a78ffb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
